In [24]:
using LowLevelFEM
using SparseArrays
using LinearAlgebra

In [25]:
gmsh.initialize()

In [26]:
structured_rect_mesh(lx=2, order=2)

In [27]:
material = Material("surface")

Pp = Problem([material], type=:ScalarField, field=:p, rhs_field=:fp, dim=2)
Pu = Problem([material], type=:VectorField, field=:v, rhs_field=:fu, dim=2)

Problem("", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fu)

In [28]:
presL = BoundaryCondition("left", problem=Pp, p=10)
presR = BoundaryCondition("right", problem=Pp, p=1)
pres0 = BoundaryCondition("rightbottom", problem=Pp, p=0)

BoundaryCondition("rightbottom", Problem("", :ScalarField, 2, 1, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp), Dict{Symbol, Union{Function, Number, ScalarField}}(:p => 0))

In [29]:
suppT = BoundaryCondition("top", problem=Pu, vx=0, vy=0)
suppB = BoundaryCondition("bottom", problem=Pu, vx=0, vy=0)

BoundaryCondition("bottom", Problem("", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fu), Dict{Symbol, Union{Function, Number, ScalarField}}(:vy => 0, :vx => 0))

In [30]:
fu = loadVector(Pu, [])
fp = loadVector(Pp, [])

nodal ScalarField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [31]:
load_u = LoadCondition("surface", fux=1.0, fuy=0.0)
fu = loadVector(Pu, [load_u])

# g = 0 a nyomásegyenletben
gp = loadVector(Pp, [])   # vagy csinálj null ScalarField-et, ahogy nálad kényelmes

F = SystemVector([fu, gp])

SystemVector([0.0002777777777777773, 0.0, 0.0002777777777777746, 0.0, 0.0002777777777777775, 0.0, 0.00027777777777777767, 0.0, 0.0005555555555555548, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], nothing, Problem[Problem("", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fu), Problem("", :ScalarField, 2, 1, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp)], [0, 1722])

In [32]:
μ = 1.0

# stabilizációk (kezdésnek)
γ = 1e-1          # grad-div (tipikusan 1e-2...1e0)
δ = 1e-6          # pressure Laplacian (meshfüggő lenne; induljunk kicsivel)

A = ∫((SymGrad(Pu) ⋅ SymGrad(Pu)) * 2μ)
D = ∫(Div(Pu) ⋅ Div(Pu) * γ)

# Egyben is meg lehetne adni
AD = ∫((SymGrad(Pu) ⋅ SymGrad(Pu)) * 2μ + γ * (Div(Pu) ⋅ Div(Pu)))

B = ∫(Div(Pu) ⋅ Id(Pp))
C = ∫(Grad(Pp) ⋅ Grad(Pp) * δ)

# nagy blokk: [AD   B';  B   -C]
K = SystemMatrix([A+D B';
    B -C])

sparse([1, 2, 9, 10, 47, 48, 219, 220, 239, 240  …  1722, 1725, 1774, 1784, 1785, 1804, 2013, 2563, 2581, 2583], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  2583, 2583, 2583, 2583, 2583, 2583, 2583, 2583, 2583, 2583], [1.8666666666666698, 1.0000000000000044, 0.02222222222222278, -5.551115123125783e-17, -1.1111111111111125, -1.7763568394002505e-15, -0.2222222222222221, -3.0531133177191805e-16, -0.08888888888888558, 2.220446049250313e-16  …  2.42861286636753e-17, 0.7111111111111112, 0.7111111111111188, 2.133333333333322, 0.7111111111111057, 2.133333333333322, 0.7111111111111028, 2.1333333333333373, 2.1333333333333457, -11.377777777777766], 2583, 2583)

In [33]:
typeof(A)


SystemMatrix

In [34]:
typeof(D)

SystemMatrix

In [35]:
A + D

sparse([1, 2, 9, 10, 47, 48, 219, 220, 239, 240  …  163, 164, 581, 582, 1681, 1682, 1717, 1718, 1721, 1722], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  1722, 1722, 1722, 1722, 1722, 1722, 1722, 1722, 1722, 1722], [1.8666666666666698, 1.0000000000000044, 0.02222222222222278, -5.551115123125783e-17, -1.1111111111111125, -1.7763568394002505e-15, -0.2222222222222221, -3.0531133177191805e-16, -0.08888888888888558, 2.220446049250313e-16  …  9.936496070395151e-15, -4.977777777777758, -1.7777777777777635, -1.0666666666666536, 2.1788126858268697e-15, -1.4222222222222283, -2.268324417187273e-14, -4.977777777777796, -8.382183835919932e-15, 17.066666666666652], 1722, 1722)

In [36]:
@showfields A

SystemMatrix:
  A = sparse([1, 2, 9, 10, 47, 48, 219, 220, 239, 240, 241, 242, 583, 584, 585, 586, 587, 588, 1, 2, 9, 10, 47, 48, 219, 220, 239, 240, 241, 242, 583, 584, 585, 586, 587, 588, 3, 4, 45, 46, 85, 86, 87, 88, 105, 106, 565, 566, 1627, 1628, 1685, 1686, 1687, 1688, 3, 4, 45, 46, 85, 86, 87, 88, 105, 106, 565, 566, 1627, 1628, 1685, 1686, 1687, 1688, 5, 6, 103, 104, 123, 124, 125, 126, 163, 164, 581, 582, 1681, 1682, 1717, 1718, 1721, 1722, 5, 6, 103, 104, 123, 124, 125, 126, 163, 164, 581, 582, 1681, 1682, 1717, 1718, 1721, 1722, 7, 8, 161, 162, 201, 202, 203, 204, 221, 222, 257, 258, 633, 634, 637, 638, 639, 640, 7, 8, 161, 162, 201, 202, 203, 204, 221, 222, 257, 258, 633, 634, 637, 638, 639, 640, 1, 2, 9, 10, 11, 12, 47, 48, 49, 50, 219, 220, 239, 240, 241, 242, 259, 260, 583, 584, 585, 586, 587, 588, 641, 642, 643, 644, 645, 646, 1, 2, 9, 10, 11, 12, 47, 48, 49, 50, 219, 220, 239, 240, 241, 242, 259, 260, 583, 584, 585, 586, 587, 588, 641, 642, 643, 644, 645, 646, 9, 10, 1

, 1685, 1686, 1689, 1690, 1691, 1692, 1693, 1694, 1695, 1696, 87, 88, 89, 90, 91, 92, 107, 108, 109, 110, 565, 566, 567, 568, 569, 570, 1633, 1634, 1639, 1640, 1685, 1686, 1689, 1690, 1691, 1692, 1693, 1694, 1695, 1696, 89, 90, 91, 92, 93, 94, 109, 110, 111, 112, 567, 568, 569, 570, 571, 572, 1639, 1640, 1645, 1646, 1689, 1690, 1693, 1694, 1695, 1696, 1697, 1698, 1699, 1700, 89, 90, 91, 92, 93, 94, 109, 110, 111, 112, 567, 568, 569, 570, 571, 572, 1639, 1640, 1645, 1646, 1689, 1690, 1693, 1694, 1695, 1696, 1697, 1698, 1699, 1700, 91, 92, 93, 94, 95, 96, 111, 112, 113, 114, 569, 570, 571, 572, 573, 574, 1645, 1646, 1651, 1652, 1693, 1694, 1697, 1698, 1699, 1700, 1701, 1702, 1703, 1704, 91, 92, 93, 94, 95, 96, 111, 112, 113, 114, 569, 570, 571, 572, 573, 574, 1645, 1646, 1651, 1652, 1693, 1694, 1697, 1698, 1699, 1700, 1701, 1702, 1703, 1704, 93, 94, 95, 96, 97, 98, 113, 114, 115, 116, 571, 572, 573, 574, 575, 576, 1651, 1652, 1657, 1658, 1697, 1698, 1701, 1702, 1703, 1704, 1705, 1706, 17

Excessive output truncated after 524291 bytes.

, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1474, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1475, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1476, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1477, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1478, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1479, 1480, 1480, 1480, 1480, 1480, 1480, 1480, 1480, 1480, 1480, 14

In [37]:
C = ∫(Γ="right", Id(Pu) ⋅ Id(Pu), weight=[1 0; 0 1])
C[:, :]

1722×1722 SparseMatrixCSC{Float64, Int64} with 162 stored entries:
⎡⠁⠀⠉⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠇⠀⠻⠆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [38]:
C = ∫(Γ="right", Id(Pu) ⋅ [1 0; 0 1] ⋅ Id(Pu))
C[:, :]

1722×1722 SparseMatrixCSC{Float64, Int64} with 162 stored entries:
⎡⠁⠀⠉⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠇⠀⠻⠆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [39]:
s0 = scalarField(Pu, "surface", 0)
s1 = scalarField(Pu, "surface", 1)
C = ∫(Γ="right", ε(Pu) ⋅ [s1 s0; s0 s1] ⋅ ε(Pu))
C[:, :]

1722×1722 SparseMatrixCSC{Float64, Int64} with 162 stored entries:
⎡⠁⠀⠉⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠇⠀⠻⠆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [40]:
s1 = scalarField(Pu, "surface", 1)
C = ∫(Γ="right", s1 * (Id(Pu) ⋅ Id(Pu)))
C[:, :]

1722×1722 SparseMatrixCSC{Float64, Int64} with 162 stored entries:
⎡⠁⠀⠉⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠇⠀⠻⠆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [41]:
C = ∫(Ω="surface", Grad(Pu) ⋅ Grad(Pu))
C[:, :]

1722×1722 SparseMatrixCSC{Float64, Int64} with 26082 stored entries:
⎡⢿⣷⠭⠉⠈⠽⠿⣖⣒⣒⣶⣦⠤⠽⠸⠇⠇⢰⢰⡆⡂⢐⢐⡂⡂⢐⢐⡂⡆⢰⢰⡆⡄⢠⠠⠄⠄⠨⠨⠈⎤
⎢⡇⠃⢻⣶⣄⡀⣀⣀⣀⣠⡤⠬⠭⢿⢀⠀⡀⣀⢀⠀⡀⣀⢀⠀⡀⣀⢠⠀⡄⠤⠠⠀⠅⠬⠨⠁⠅⠼⢻⢻⎥
⎢⣆⡄⠀⠹⢱⣶⡖⠒⠒⠋⠉⠉⠉⢩⣶⣀⠂⠒⠐⠀⠂⠒⠐⠀⠃⠉⠈⠀⠁⠉⠈⠀⠁⠉⠈⠀⠁⠉⠈⠀⎥
⎢⢻⢧⠀⢸⢸⠉⠻⣦⡀⠀⠀⠀⠀⠈⠙⠛⠿⢷⣶⣦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢸⢸⠀⣸⡼⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠛⠻⠿⣶⣦⣤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠸⣿⡀⡏⡇⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠛⠿⢷⣶⣤⣤⣀⡀⠀⠀⠀⠀⎥
⎢⣄⡇⣧⣇⡇⣀⡀⠀⠀⠀⠀⠈⠻⣦⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠛⠻⠿⣶⣦⣤⎥
⎢⠶⠆⠀⠐⠘⢻⣷⠀⠀⠀⠀⠀⠀⠸⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢉⣁⠀⢨⢨⠀⢿⣇⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠰⠶⠀⠐⠐⠀⠸⣿⡄⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢈⢈⠀⢨⢨⠀⠀⢻⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠰⠰⠀⠐⠐⠀⠀⠈⣿⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢈⢈⠀⢨⡍⠀⠀⠀⠸⣿⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠰⠰⠀⠒⠂⠀⠀⠀⠀⢿⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢈⣉⠀⡍⡅⠀⠀⠀⠀⠘⣿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠰⠶⠀⠂⠂⠀⠀⠀⠀⠀⢹⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⣉⡁⡅⡅⠀⠀⠀⠀⠀⠀⣿⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⎥
⎢⠀⠆⠆⠂⠂⠀⠀⠀⠀⠀⠀⠸⣿⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⎥
⎢⡀⡁⣁⡅⡅⠀⠀⠀⠀⠀⠀⠀⢻⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⎥
⎣⡂⠂⣿⣒⠂⠀⠀⠀⠀⠀⠀⠀⠈⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⎦

In [42]:
K[:, :]

2583×2583 SparseMatrixCSC{Float64, Int64} with 116825 stored entries:
⎡⣿⢟⣉⠙⠿⠷⠶⠶⢾⡋⠯⠽⠸⠸⠶⠆⠆⠆⠶⠰⠰⠰⠶⠆⡆⣋⣻⣏⠻⠷⢾⡿⠿⠷⠶⠶⠶⠶⢶⣋⎤
⎢⣇⠘⢿⣷⡶⠾⠿⠛⠛⣷⡶⠰⠰⠶⠆⠇⠇⠭⠽⠘⠘⠛⠃⠃⠃⠛⢹⢻⣷⠿⠛⣷⠶⠶⠿⠽⠛⠛⠛⠛⎥
⎢⢿⡇⣸⡏⠻⣦⡀⠀⠀⠉⠛⠻⠶⣶⣤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣾⡿⣆⠀⠙⠷⣦⣄⠀⠀⠀⠀⠀⎥
⎢⢸⡇⣿⠃⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠈⠉⠙⠛⠷⢶⣦⣤⣀⡀⠀⠀⢸⣿⠁⠹⣆⠀⠀⠈⠙⠻⣦⣄⡀⠀⎥
⎢⡾⠳⢿⣤⡄⠀⠀⠈⠻⣦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠛⠻⠶⢾⢿⣤⠀⠹⣦⠀⠀⠀⠀⠀⠙⠻⠶⎥
⎢⣏⡇⢘⡋⣿⡀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣘⣻⡀⠀⢿⣧⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣒⡂⢰⡆⢸⣧⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢐⣲⣾⡇⠀⠀⢻⣧⠀⠀⠀⠀⠀⠀⎥
⎢⠸⠇⠬⠅⠀⢿⡆⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⠯⠅⣧⠀⠀⠀⢿⣧⠀⠀⠀⠀⠀⎥
⎢⠨⠅⡍⡅⠀⠘⣷⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠨⢭⠅⢻⠀⠀⠀⠀⢻⣧⠀⠀⠀⠀⎥
⎢⢘⡃⣓⠃⠀⠀⢹⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⢘⣛⠀⢸⡇⠀⠀⠀⠀⢻⣧⠀⠀⠀⎥
⎢⢐⡂⣶⠀⠀⠀⠈⣿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⢐⣲⠀⠈⣇⠀⠀⠀⠀⠀⢻⣧⠀⠀⎥
⎢⠸⠇⠭⠀⠀⠀⠀⠸⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠸⠯⠀⠀⣿⠀⠀⠀⠀⠀⠈⢻⣧⠀⎥
⎢⡬⢩⣭⠀⠀⠀⠀⠀⢻⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣬⣭⠀⠀⢸⡆⠀⠀⠀⠀⠀⠀⢻⣧⎥
⎢⡿⢾⣷⣒⣲⣶⣶⣶⣾⣗⣒⢲⢰⣰⡶⡆⡆⣆⣶⢰⢰⣰⡶⡆⡆⣿⣿⣿⣲⣶⣾⣗⣶⣶⣶⣶⣶⣶⣶⣿⎥
⎢⢿⡆⣽⡟⠻⢯⣅⡀⠀⠛⠛⠺⠾⠿⠥⣥⣥⣁⣀⣀⡀⠀⠀⠀⠀⠀⢸⣾⡿⣯⡀⠛⠿⠯⣭⣁⣀⠀⠀⠀⎥
⎢⣾⡷⢿⣤⣄⠀⠈⠙⠳⣦⣤⣄⠀⠀⠀⠀⠀⠀⠉⠉⠉⠙⠛⠛⠲⠶⢾⢿⣤⠈⠻⣦⡀⠀⠀⠈⠉⠛⠳⠶⎥
⎢⢿⡇⢸⡇⠹⣧⡀⠀⠀⠀⠉⠛⠿⣶⣤⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⡿⡇⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⢸⡇⣟⡇⠀⠙⣷⡀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣤⣀⠀⠀⠀⠀⠀⠀⢸⣿⠇⢻⡀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⢸⡇⣿⠀⠀⠀⠈⢿⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣦⣀⠀⠀⢸⣿⠀⠘⣧⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⡼⢳⣿⠀⠀⠀⠀⠈⢻⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣼⣿⠀⠀⢹⡆⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [43]:
#F = SystemVector([fu, fp])

In [44]:
u, p = solveField(K, F, support=[pres0, suppT, suppB])

(VectorField(Matrix{Float64}[], [0.0; 0.0; … ; 0.011297741328270555; 0.00629789381568244;;], [0.0], Int64[], 1, :v2D, Problem("", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fu)), ScalarField(Matrix{Float64}[], [-0.0015188347053337822; 0.0; … ; 0.0015024261747974184; 0.0008788976825677312;;], [0.0], Int64[], 1, :scalar, Problem("", :ScalarField, 2, 1, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp)))

In [45]:
showDoFResults(u, name="u", visible=true)
showDoFResults(p, name="p")

openPostProcessor()

In [46]:
gmsh.finalize()